# NanoGPT speedrun, round 3: does Canon survive on a record-like base?

**What this tests, and why.** In rounds 1-2 of this one-GPU proxy, Canon layers (Allen-Zhu 2025: a causal
per-channel 4-tap convolution with a residual, before attention and before the MLP; zero-init here) were the
biggest new win: -207 millinats (mnat) of val loss on the plain baseline, and -48 to -92 on a stack of record
techniques. But those bases lacked two record features that overlap Canon: the **partial key offset** (each key's
non-rotating dims come from the previous token, a hard-coded one-tap Canon on the keys, PR #169) and the
**hashed n-gram table**. This notebook re-tests on **REC**, a record-like base: Polar Express, NorMuon, cautious
weight decay, value embeddings, x0 mix, attention gate, smear and the key offset. It asks:

1. **Does Canon survive?** `REC+canon` against `REC`, and `REC+trigram_hash+canon` against `REC+trigram_hash`.
2. **Which form?** Sites: `canon_a` (before attention only) and `canon_c` (before the MLP only). Kernel size:
   `canon_k2` and `canon_k3`. Placement of the norm: `canon_prenorm` is norm(x + conv(x)) and `canon_renorm` is
   norm(n + conv(n)) with n = norm(x); these two fit the record's static fp8 scales. `canon_from1` skips layer 0.
   `canon_bk` is a learned key offset: Canon on the keys of every layer, starting as the record's hard offset where
   that is on. `v_shift` is Canon on the values.
3. **Is it cheap?** Steady-state ms/step against REC, with compile and warmup excluded.

`REC+rec_rep` re-runs REC under the same seeds, so its differences measure the GPU's run-to-run nondeterminism.
That is the floor under every comparison. With `RUN_LONG`, REC and REC+canon also run at 3x the steps, to see
whether the gain shrinks with more tokens.

**Fixed since rounds 1-2.** Their losses stand, but their time column does not: after 8 distinct configs, dynamo
silently ran every new config eagerly, which made it 2.7-3.5x slower. Each config now compiles fresh, and step time
is measured in the steady state. Under one seed every arm now starts from the same init of every shared parameter,
so arms are compared per seed. A tail EMA of the weights is evaluated next to the final weights.

**How to run:**
1. Upload this notebook to Colab (File → Upload notebook).
2. Runtime → Change runtime type → the biggest GPU offered (RTX PRO 6000, H100 or A100).
3. Runtime → **Run all**.

It needs no code from GitHub. The data, about 400 MB of the speedrun's GPT-2-tokenized FineWeb, comes from
Hugging Face.

**Two GPUs:** open a second copy on the other GPU and set `FIRST_SEED = 2` in cell 2. It runs seeds 2 and 3 of the
same arms, and the two summaries pool into 4 seeds. To see the pooled verdict, paste one copy's summary into the
last cell of the other.

**Time per copy:** the main cell is 15 arms × 2 seeds = 30 runs of 1200 steps (20M tokens each). At about 50 s per
run on an RTX PRO 6000 or A100, plus up to a minute of compile per arm, that is **about 35-40 minutes**. The
long-horizon cell (2 arms × 2 seeds × 3600 steps) adds **about 12 minutes**. An H100 is faster. Every finished run
is appended to a results file and **re-running skips finished runs**: after a disconnect, Run all resumes where it
stopped. Colab wipes `/content` when it recycles a runtime, so set `SAVE_TO_DRIVE = True` to keep the results
through that. The most important arms run first.

**What to paste back to Claude:** the **Summary** cell's output (one JSON line per arm) from each copy. The verdict
table is welcome too.

A proxy win is a screen, not proof: the record is ~124M params plus an 84.6M-row n-gram table, trained on ~330M
tokens with fp8 matmuls.

In [ ]:
# 1. GPU check, and the one package Colab lacks (PyTorch and NumPy are preinstalled).
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q huggingface_hub

In [ ]:
# 2. Settings
SEEDS = 2           # seeds per arm in this copy. Same seed = same init of every shared parameter (paired)
FIRST_SEED = 0      # in a second copy (another GPU) set 2: it runs seeds 2 and 3, and the two summaries pool
RUN_LONG = True     # also REC vs REC+canon at 3x the steps (~12 more minutes): does the gain shrink with tokens?
DATA_DIR = "/content/fineweb10B"
RESULTS_FILE = "/content/canon_results.jsonl"  # every finished run is appended; re-running skips them
SAVE_TO_DRIVE = False  # True: keep results in Google Drive across disconnects (asks for permission)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_FILE = f"/content/drive/MyDrive/canon_results_s{FIRST_SEED}.jsonl"

In [ ]:
# 3. Data: the speedrun's own GPT-2-tokenized FineWeb from Hugging Face: the val shard and 1 of the 103 train
# shards (~400 MB). One run reads 20M tokens.
import os
from huggingface_hub import hf_hub_download

os.makedirs(DATA_DIR, exist_ok=True)
for name in ("fineweb_val_000000.bin", "fineweb_train_000001.bin"):
    if not os.path.exists(os.path.join(DATA_DIR, name)):
        hf_hub_download(repo_id="kjj0/fineweb10B-gpt2", filename=name, repo_type="dataset", local_dir=DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

In [ ]:
# 4. The proxy: model, optimizers, training loop, and the ideas as flags (VARIANTS).
import argparse
import dataclasses
import glob
import json
import math
import time
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch._dynamo.utils  # counters: graphs compiled per run

# bf16 autocast where the GPU has bf16 tensor cores (A100, L4, H100); fp32 elsewhere (CPU, T4).
AMP = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
BOS = 50256
VOCAB = 50304  # 50257 padded to a multiple of 128, as the record


@dataclass
class Config:
    # scale (the default fits a 16 GB GPU and runs a few minutes on an A100)
    n_layer: int = 8
    d_model: int = 512
    head_dim: int = 128
    seq_len: int = 1024
    batch_seqs: int = 16          # 16k tokens/step: the fp32 logits stay ~3 GB
    steps: int = 1200
    val_tokens: int = 2 * 1024 * 1024
    # optimization
    lr_hidden: float = 0.03       # Muon
    lr_embed: float = 0.3         # Adam: embeddings
    lr_head: float = 0.008        # Adam: lm_head
    lr_scalar: float = 0.02       # Adam: scalars and small vectors
    momentum: float = 0.95
    weight_decay: float = 0.0
    cooldown_frac: float = 0.5    # decay to lr_floor over the last fraction of steps
    lr_floor: float = 0.1
    cooldown_power: float = 1.0   # 1 = linear; >1 decays faster early in the cooldown ("PowerCool")
    snoo_every: int = 0           # >0: Snoo-style outer Nesterov step on slow weights every this many steps
    snoo_lr: float = 0.7
    snoo_momentum: float = 0.5
    warmup_steps: int = 0
    # ideas (flags)
    orth: str = "ns5"             # ns5 | polar (Polar Express) | none (plain momentum SGD, normalized)
    normuon: bool = False         # per-row second-moment normalization after orthogonalization
    cautious_wd: bool = False     # weight decay only where update and weight agree in sign
    hidden_opt: str = "muon"      # muon | adam
    softcap: float = 15.0
    act: str = "relu2"            # relu2 | gelu | swiglu
    value_embeds: int = 0         # number of value-embedding tables mixed into V of the first/last layers
    unet: bool = False            # skip connections from the first half of the layers to the second half
    x0_mix: bool = False          # learnable mix of the embedding back into every block's input
    smear: bool = False           # gated 1-token look-back on the embeddings
    attn_gate: bool = False       # per-head sigmoid output gate from the block input
    bigram_rows: int = 0          # hashed bigram embedding table rows (0 = off)
    trigram_rows: int = 0         # hashed trigram embedding table rows (0 = off)
    ngram_gate: bool = False      # content-aware sigmoid gate on the hashed n-gram embeddings (Engram-style)
    mtp: float = 0.0              # weight of a t+2 prediction loss through the same head
    zloss: float = 0.0            # weight of the softmax normalizer z-loss
    qk_norm: bool = True
    canon: bool = False           # Canon layers: zero-init causal per-channel conv + residual on the sublayer inputs
    canon_k: int = 4              # Canon kernel size (taps: the token itself and k-1 previous ones)
    canon_sites: str = "AC"       # A: before attention, C: before the MLP
    canon_norm: str = "post"      # post: conv(norm(x)) (the paper's); pre: norm(conv(x)); renorm: norm(conv(norm(x)))
    canon_from_layer: int = 0     # no Canon on the layers below this one
    canon_bk: bool = False        # Canon-B on the normed keys (k=4, every layer); starts as the key offset where on
    v_shift: bool = False         # Canon-B on the values (k=4, zero-init, every layer)
    key_offset: bool = False      # the record's partial key offset (#169): stationary key dims from the previous token
    key_offset_layers: tuple = (2, 7)  # the record's long-window layers (3, 10 of 11), for 8 layers
    diff_attn: bool = False       # differential attention: softmax(q1 k1) - lambda softmax(q2 k2) on half-width q, k
    attn_scale: float = 0.0       # 0 = 1/sqrt(head_dim)
    tie_embed: bool = False
    # measurement
    tail_ema: float = 0.98        # >0: also evaluate an EMA of the weights over the last tail_frac of steps
    tail_frac: float = 0.2
    timing_warmup: int = 50       # steps excluded from the steady-state ms/step (compile, autotune, allocator)
    seed: int = 0
    compile: bool = False

    def update(self, **kw):
        return dataclasses.replace(self, **kw)


# Each variant: what it changes relative to `baseline`. Add ideas here; keep them one flag or a few each.
VARIANTS = {
    "baseline": {},
    "rec_rep": {},  # no change: X+rec_rep re-runs X under the same seeds (its own run key): GPU nondeterminism
    "polar_express": {"orth": "polar"},
    "normuon": {"normuon": True},
    "cautious_wd": {"cautious_wd": True, "weight_decay": 0.025},
    "plain_wd": {"weight_decay": 0.025},
    "adam_hidden": {"hidden_opt": "adam", "lr_hidden": 0.003},
    "gelu": {"act": "gelu"},
    "swiglu": {"act": "swiglu"},
    "value_embeds": {"value_embeds": 2},
    "unet": {"unet": True},
    "x0_mix": {"x0_mix": True},
    "smear": {"smear": True},
    "attn_gate": {"attn_gate": True},
    "bigram_hash": {"bigram_rows": 1 << 18},
    "mtp": {"mtp": 0.3},
    "zloss": {"zloss": 1e-4},
    "softcap30": {"softcap": 30.0},
    "no_qk_norm": {"qk_norm": False},
    "cooldown_0.8": {"cooldown_frac": 0.8},
    "tied": {"tie_embed": True, "lr_embed": 0.01},  # the shared matrix is the head too: a head-sized lr
    "momentum_0.9": {"momentum": 0.9},
    "lr_hidden_x1.5": {"lr_hidden": 0.045},
    "powercool": {"cooldown_power": 2.0},
    "snoo": {"snoo_every": 8},
    "trigram_hash": {"bigram_rows": 1 << 18, "trigram_rows": 1 << 18},
    "ngram_gate": {"bigram_rows": 1 << 18, "ngram_gate": True},
    "canon": {"canon": True},
    "diff_attn": {"diff_attn": True},
    "key_offset": {"key_offset": True},
    "canon_a": {"canon": True, "canon_sites": "A"},
    "canon_c": {"canon": True, "canon_sites": "C"},
    "canon_k2": {"canon": True, "canon_k": 2},
    "canon_k3": {"canon": True, "canon_k": 3},
    "canon_prenorm": {"canon": True, "canon_norm": "pre"},   # the fp8-safe forms for the record (its static scales
    "canon_renorm": {"canon": True, "canon_norm": "renorm"},  # assume normed inputs)
    "canon_from1": {"canon": True, "canon_from_layer": 1},
    "canon_bk": {"canon_bk": True},
    "v_shift": {"v_shift": True},
}


# ------------------------------------------------------------------------------------------------ data

def load_tokens(path: str, limit: int | None = None) -> np.ndarray:
    header = np.fromfile(path, dtype=np.int32, count=256)
    assert header[0] == 20240520 and header[1] == 1, f"{path}: not a FineWeb .bin shard"
    n = int(header[2]) if limit is None else min(int(header[2]), limit)
    return np.memmap(path, dtype=np.uint16, mode="r", offset=256 * 4, shape=(n,))


class TrainStream:
    """Consecutive windows of seq_len + 1 tokens through the train shards, in file order: every variant and
    every seed reads the same tokens (seeds change the init only)."""

    def __init__(self, pattern: str, seq_len: int):
        self.files = sorted(glob.glob(pattern))
        assert self.files, f"no shards match {pattern}"
        self.seq_len, self.file, self.pos = seq_len, 0, 0
        self.tokens = load_tokens(self.files[0])

    def next(self, batch_seqs: int) -> torch.Tensor:
        n = batch_seqs * self.seq_len + 1
        if self.pos + n > len(self.tokens):
            self.file = (self.file + 1) % len(self.files)
            self.tokens, self.pos = load_tokens(self.files[self.file]), 0
        chunk = torch.from_numpy(self.tokens[self.pos:self.pos + n].astype(np.int64))
        self.pos += n - 1
        return chunk


# ------------------------------------------------------------------------------------------------ model

def norm(x):
    return F.rms_norm(x, (x.size(-1),))


class Rotary(nn.Module):
    def __init__(self, dim: int, max_len: int):
        super().__init__()
        freqs = (1 / 1024) ** torch.linspace(0, 1, dim // 4)
        freqs = torch.cat([freqs, freqs.new_zeros(dim // 4)])  # half the dims unrotated (as the record)
        theta = torch.outer(torch.arange(max_len, dtype=torch.float32), freqs)
        self.register_buffer("cos", theta.cos(), persistent=False)
        self.register_buffer("sin", theta.sin(), persistent=False)
        # dim i pairs with i + dim/2: the stationary dims are [dim/4, dim/2) and [3 dim/4, dim)
        self.register_buffer("stationary", torch.cat([freqs, freqs]) == 0, persistent=False)

    def forward(self, x):  # [B, T, H, D]
        cos, sin = self.cos[None, :x.size(1), None, :].type_as(x), self.sin[None, :x.size(1), None, :].type_as(x)
        x1, x2 = x.float().chunk(2, dim=-1)
        x1, x2 = x1.type_as(x), x2.type_as(x)
        return torch.cat((x1 * cos + x2 * sin, -x1 * sin + x2 * cos), dim=-1)


def canon(w: torch.Tensor, h: torch.Tensor, edge: bool = False) -> torch.Tensor:
    """Canon layer as shift-and-sum: h[t] + sum_j w[j] * h[t - j], j = 0..k-1. Before the sequence start the taps
    read zeros: a causal depthwise Conv1d whose weight is w.T.flip(-1)[:, None, :], with k-1 left padding. With
    edge=True they read the first token instead, as the key offset does (token 0 keeps its own key).
    w: [k, d] per-channel taps (w[0] weighs the token itself); h: [B, T, d]. Pointwise ops only: inductor fuses it
    into one kernel."""
    out = h + w[0] * h
    for j in range(1, w.size(0)):
        out = out + w[j] * (torch.cat([h[:, :1].expand(-1, j, -1), h[:, :-j]], dim=1) if edge
                            else F.pad(h[:, :-j], (0, 0, j, 0)))
    return out.type_as(h)


def key_offset(k: torch.Tensor, stationary: torch.Tensor) -> torch.Tensor:
    """The record's partial key offset (#169): every key's stationary dims come from the previous token's key
    (token 0 keeps its own). k: [B, T, H, D], normed; stationary: [D] bool."""
    return torch.where(stationary, torch.cat([k[:, :1], k[:, :-1]], dim=1), k)


class Block(nn.Module):
    def __init__(self, cfg: Config, layer: int):
        super().__init__()
        d, hd = cfg.d_model, cfg.head_dim
        self.cfg, self.layer, self.n_head = cfg, layer, d // hd
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)
        hidden = 4 * d if cfg.act != "swiglu" else int(8 * d / 3) // 64 * 64
        self.fc = nn.Linear(d, hidden * (2 if cfg.act == "swiglu" else 1), bias=False)
        self.out = nn.Linear(hidden, d, bias=False)
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.out.weight)
        self.rotary = Rotary(hd, cfg.seq_len)
        self.lambdas = nn.Parameter(torch.tensor([1.0, 0.0]))  # (x, x0) mix, used with x0_mix
        self.ve_lambda = nn.Parameter(torch.tensor(0.5))
        # Optional parameters are built RNG-free (torch.zeros, never nn.Linear/nn.Conv1d + zeros_, whose constructors
        # draw from the global RNG): every shared parameter then gets the same init under one seed in every variant.
        if cfg.attn_gate:
            self.attn_gate_w = nn.Parameter(torch.zeros(self.n_head, 12))
        canon_here = cfg.canon and layer >= cfg.canon_from_layer  # depthwise, causal, zero-init: starts as before
        self.canon_a = nn.Parameter(torch.zeros(cfg.canon_k, d)) if canon_here and "A" in cfg.canon_sites else None
        self.canon_c = nn.Parameter(torch.zeros(cfg.canon_k, d)) if canon_here and "C" in cfg.canon_sites else None
        self.canon_bv = nn.Parameter(torch.zeros(4, d)) if cfg.v_shift else None
        self.key_offset = cfg.key_offset and layer in cfg.key_offset_layers
        self.canon_bk = None
        if cfg.canon_bk:  # replaces the hard key offset, starting as it (stationary dims: k - k + k[t-1])
            w = torch.zeros(4, self.n_head, hd)
            if self.key_offset:
                w[0, :, self.rotary.stationary], w[1, :, self.rotary.stationary] = -1.0, 1.0
            self.canon_bk, self.key_offset = nn.Parameter(w.view(4, d)), False
        assert not (cfg.diff_attn and (cfg.key_offset or cfg.canon_bk)), "key offset, canon_bk: not with diff_attn"
        if cfg.diff_attn:
            self.rotary_half = Rotary(hd // 2, cfg.seq_len)
            self.diff_lambda = nn.Parameter(torch.tensor(0.2))

    def sublayer_input(self, x, w):
        """norm(x), through the Canon layer w (if any) where canon_norm puts it."""
        if w is None:
            return norm(x)
        if self.cfg.canon_norm == "pre":
            return norm(canon(w, x))
        h = canon(w, norm(x))
        return norm(h) if self.cfg.canon_norm == "renorm" else h

    def attend(self, q, k, v):
        """[B, T, H, D] -> [B, T, H, D] causal attention (differential if configured)."""
        scale = self.cfg.attn_scale or None
        if not self.cfg.diff_attn:
            if self.cfg.qk_norm:
                q, k = norm(q), norm(k)
            if self.canon_bk is not None:  # Canon-B on the keys, before the rotary (as the paper's)
                k = canon(self.canon_bk, k.flatten(2), edge=True).view_as(k)
            elif self.key_offset:  # the stationary dims are the same before and after the rotary
                k = key_offset(k, self.rotary.stationary)
            q, k = self.rotary(q), self.rotary(k)
            return F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2),
                                                  is_causal=True, scale=scale).transpose(1, 2)
        ys = []
        for qh, kh in zip(q.chunk(2, dim=-1), k.chunk(2, dim=-1)):
            if self.cfg.qk_norm:
                qh, kh = norm(qh), norm(kh)
            qh, kh = self.rotary_half(qh), self.rotary_half(kh)
            ys.append(F.scaled_dot_product_attention(qh.transpose(1, 2), kh.transpose(1, 2), v.transpose(1, 2),
                                                     is_causal=True, scale=scale).transpose(1, 2))
        return norm(ys[0] - self.diff_lambda * ys[1]) * (1 - 0.2)  # per-head norm, scaled as in the paper

    def forward(self, x, x0, ve):
        if self.cfg.x0_mix:
            x = self.lambdas[0] * x + self.lambdas[1] * x0
        B, T, _ = x.shape
        h = self.sublayer_input(x, self.canon_a)
        q, k, v = self.qkv(h).view(B, T, 3, self.n_head, -1).unbind(2)
        if self.canon_bv is not None:
            v = canon(self.canon_bv, v.flatten(2)).view_as(v)
        if ve is not None:
            v = self.ve_lambda * v + (1 - self.ve_lambda) * ve.view_as(v)
        y = self.attend(q, k, v)
        if self.cfg.attn_gate:
            y = y * torch.sigmoid(F.linear(h[..., :12], self.attn_gate_w) + 2.0)[..., None]  # starts near open
        x = x + self.proj(y.reshape(B, T, -1))
        h = self.fc(self.sublayer_input(x, self.canon_c))
        if self.cfg.act == "relu2":
            h = F.relu(h).square()
        elif self.cfg.act == "gelu":
            h = F.gelu(h)
        else:
            a, g = h.chunk(2, dim=-1)
            h = F.silu(g) * a
        return x + self.out(h)


class GPT(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(VOCAB, cfg.d_model)
        assert not cfg.key_offset or max(cfg.key_offset_layers) < cfg.n_layer, "key_offset_layers out of range"
        assert set(cfg.canon_sites) <= {"A", "C"} and cfg.canon_norm in ("post", "pre", "renorm"), "unknown Canon form"
        self.blocks = nn.ModuleList(Block(cfg, i) for i in range(cfg.n_layer))
        self.head = nn.Linear(cfg.d_model, VOCAB, bias=False)
        nn.init.zeros_(self.head.weight)
        if cfg.tie_embed:
            nn.init.normal_(self.embed.weight, std=0.02)  # it is also the head now: small logits at init
            self.head.weight = self.embed.weight
        # Random optional tables draw from their own stream (seeded by cfg.seed), never from the global RNG.
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(cfg.seed + 1_000_003)
            self.ve = nn.ModuleList(nn.Embedding(VOCAB, cfg.d_model) for _ in range(cfg.value_embeds))
        if cfg.unet:
            self.skip_w = nn.Parameter(torch.ones(cfg.n_layer // 2))
        if cfg.smear:
            self.smear_gate = nn.Parameter(torch.zeros(1, 12))
        if cfg.bigram_rows:  # zero-init tables: plain zeros (nn.Embedding would draw rows x d normals first)
            self.bigram = nn.Parameter(torch.zeros(cfg.bigram_rows, cfg.d_model))
        if cfg.trigram_rows:
            self.trigram = nn.Parameter(torch.zeros(cfg.trigram_rows, cfg.d_model))
        if cfg.ngram_gate:
            self.ngram_gate_w = nn.Parameter(torch.zeros(1, 12))

    def value_embed_for(self, layer: int, ves):
        # U-net pattern: table i feeds layer i and layer n_layer - 1 - i
        n = len(ves)
        if layer < n:
            return ves[layer]
        if layer >= self.cfg.n_layer - n:
            return ves[self.cfg.n_layer - 1 - layer]
        return None

    def forward(self, idx, targets):
        cfg = self.cfg
        x = self.embed(idx)
        if cfg.bigram_rows:
            prev = F.pad(idx[:, :-1], (1, 0), value=BOS)
            ngram = F.embedding((prev * 36313 + idx * 27191) % cfg.bigram_rows, self.bigram)
            if cfg.trigram_rows:
                prev2 = F.pad(idx[:, :-2], (2, 0), value=BOS)
                ngram = ngram + F.embedding((prev2 * 1000003 + prev * 36313 + idx * 27191) % cfg.trigram_rows, self.trigram)
            if cfg.ngram_gate:  # starts open (2 * sigmoid(0) = 1): the gate learns what to trust
                ngram = ngram * 2 * torch.sigmoid(F.linear(x[..., :12], self.ngram_gate_w))
            x = x + ngram
        if cfg.smear:
            x = torch.cat([x[:, :1], x[:, 1:] + torch.sigmoid(F.linear(x[:, 1:, :12], self.smear_gate)) * x[:, :-1]], dim=1)
        x = x0 = norm(x)
        ves = [ve(idx) for ve in self.ve]
        skips = []
        half = cfg.n_layer // 2
        for i, block in enumerate(self.blocks):
            if cfg.unet and i >= cfg.n_layer - half:
                x = x + self.skip_w[cfg.n_layer - 1 - i] * skips.pop()
            x = block(x, x0, self.value_embed_for(i, ves) if ves else None)
            if cfg.unet and i < half:
                skips.append(x)
        logits = self.head(norm(x)).float()
        logits = cfg.softcap * torch.tanh(logits / cfg.softcap)
        loss = F.cross_entropy(logits.view(-1, VOCAB), targets.reshape(-1))
        if self.training:
            if cfg.mtp:
                loss = loss + cfg.mtp * F.cross_entropy(logits[:, :-1].reshape(-1, VOCAB), targets[:, 1:].reshape(-1))
            if cfg.zloss:
                loss = loss + cfg.zloss * torch.logsumexp(logits, dim=-1).square().mean()
        return loss


# ------------------------------------------------------------------------------------------------ optimizer

NS5 = [(3.4445, -4.7750, 2.0315)] * 5
# Polar Express (Amsel et al. 2025) coefficients as used in the record (#38), with the safety factor folded in.
POLAR_EXPRESS = [
    (8.156554524902461, -22.48329292557795, 15.878769915207462),
    (4.042929935166739, -2.808917465908714, 0.5000178451051316),
    (3.8916678022926607, -2.772484153217685, 0.5060648178503393),
    (3.285753657755655, -2.3681294933425376, 0.46449024233003106),
    (2.3465413258596377, -1.7097828382687081, 0.42323551169305323),
]


def orthogonalize(G: torch.Tensor, coeffs, safety: float = 1.0) -> torch.Tensor:
    X = G.to(torch.bfloat16 if AMP else torch.float32)
    transpose = X.size(0) > X.size(1)
    if transpose:
        X = X.T
    X = X / (X.norm() * safety + 1e-7)
    for a, b, c in coeffs:
        A = X @ X.T
        X = a * X + (b * A + c * A @ A) @ X
    return (X.T if transpose else X).to(G.dtype)


class Muon(torch.optim.Optimizer):
    def __init__(self, params, cfg: Config):
        super().__init__(params, dict(lr=cfg.lr_hidden))
        self.cfg = cfg

    @torch.no_grad()
    def step(self):
        cfg = self.cfg
        coeffs = POLAR_EXPRESS if cfg.orth == "polar" else NS5
        for group in self.param_groups:
            for p in group["params"]:
                state = self.state[p]
                if not state:
                    state["buf"] = torch.zeros_like(p)
                    if cfg.normuon:
                        state["v"] = torch.zeros(p.size(0), 1, device=p.device)
                buf = state["buf"].lerp_(p.grad, 1 - cfg.momentum)
                g = p.grad.lerp(buf, cfg.momentum)  # Nesterov
                u = orthogonalize(g, coeffs, 1.01 if cfg.orth == "polar" else 1.0) if cfg.orth != "none" else g / (g.norm() + 1e-7) * math.sqrt(g.numel() / max(g.shape))
                if cfg.normuon:
                    before = u.norm()
                    state["v"].lerp_(u.square().mean(dim=1, keepdim=True), 1 - 0.95)
                    u = u / (state["v"].sqrt() + 1e-8)
                    u = u * (before / (u.norm() + 1e-8))
                u = u * max(1, p.size(0) / p.size(1)) ** 0.5
                lr = group["lr"]
                if cfg.weight_decay:
                    decay = lr * cfg.weight_decay * p
                    if cfg.cautious_wd:
                        decay = decay * (u * p > 0)
                    p.sub_(decay)
                p.add_(u.to(p.dtype), alpha=-lr)


def build_optimizers(model: GPT, cfg: Config):
    # the Canon taps (canon_a/c/bk/bv, [k, d]) stay on Adam with the scalars (lr_scalar, no weight decay), never Muon
    hidden = [p for n, p in model.named_parameters() if p.ndim == 2 and "blocks" in n and "gate" not in n
              and "canon" not in n]
    hidden_ids = {id(p) for p in hidden}
    embeds = [p for n, p in model.named_parameters() if ("embed" in n or n.startswith("ve.") or "gram" in n)
              and "gate" not in n and id(p) not in hidden_ids]
    head = [] if cfg.tie_embed else [model.head.weight]
    taken = hidden_ids | {id(p) for p in embeds + head}
    scalars = [p for p in model.parameters() if id(p) not in taken]
    adam = torch.optim.Adam([dict(params=embeds, lr=cfg.lr_embed), dict(params=head, lr=cfg.lr_head),
                             dict(params=scalars, lr=cfg.lr_scalar)], betas=(0.8, 0.95), eps=1e-10)
    if cfg.hidden_opt == "adam":
        adam.add_param_group(dict(params=hidden, lr=cfg.lr_hidden))
        opts = [adam]
    else:
        opts = [adam, Muon(hidden, cfg)]
    for opt in opts:
        for group in opt.param_groups:
            group["initial_lr"] = group["lr"]
    return opts


def lr_mult(step: int, cfg: Config) -> float:
    if step < cfg.warmup_steps:
        return (step + 1) / cfg.warmup_steps
    cd_start = int(cfg.steps * (1 - cfg.cooldown_frac))
    if step < cd_start:
        return 1.0
    t = (step - cd_start) / max(1, cfg.steps - cd_start)
    return cfg.lr_floor + (1.0 - cfg.lr_floor) * (1 - t) ** cfg.cooldown_power


# ------------------------------------------------------------------------------------------------ run

@torch.no_grad()
def evaluate(model: GPT, val: np.ndarray, cfg: Config, device) -> float:
    model.eval()
    n = cfg.val_tokens // cfg.seq_len
    losses = []
    for i in range(0, n, cfg.batch_seqs):
        rows = min(cfg.batch_seqs, n - i)
        chunk = torch.from_numpy(val[i * cfg.seq_len:(i + rows) * cfg.seq_len + 1].astype(np.int64)).to(device)
        x, y = chunk[:-1].view(rows, cfg.seq_len), chunk[1:].view(rows, cfg.seq_len)
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=AMP):
            losses.append(model(x, y).item() * rows)
    model.train()
    return sum(losses) / n


_COMPILED_FOR = None  # the config (seed aside) whose graphs dynamo holds now


def train(cfg: Config, data_dir: str, device, log_every: int = 100) -> dict:
    global _COMPILED_FOR
    if cfg.compile and _COMPILED_FOR != cfg.update(seed=0):
        # Dynamo caches compiled graphs on GPT.forward's code object across runs, guarded on the cfg values the
        # forward reads. At recompile_limit (8) distinct configs it stops compiling and runs every new config
        # eagerly (2.7-3.5x slower), silently. Start each new config from an empty cache; seeds reuse it.
        torch.compiler.reset()
        _COMPILED_FOR = cfg.update(seed=0)
    torch.manual_seed(cfg.seed)
    model = GPT(cfg).to(device)
    opts = build_optimizers(model, cfg)
    # fullgraph=True: a graph break or a recompile-limit hit raises instead of falling back to eager
    fwd = torch.compile(model, fullgraph=True) if cfg.compile else model
    graphs0 = torch._dynamo.utils.counters["stats"]["unique_graphs"]
    stream = TrainStream(f"{data_dir}/fineweb_train_*.bin", cfg.seq_len)
    # Snoo: slow weights take a Nesterov step toward the fast weights every snoo_every steps; fast restart there.
    slow = [p.detach().clone() for p in model.parameters()] if cfg.snoo_every else None
    outer_m = [torch.zeros_like(p) for p in model.parameters()] if cfg.snoo_every else None
    val = load_tokens(sorted(glob.glob(f"{data_dir}/fineweb_val_*.bin"))[0], cfg.val_tokens + 1)
    sync = torch.cuda.synchronize if device.type == "cuda" else (lambda: None)
    params = [p for p in model.parameters()]
    ema, ema_start = None, int(cfg.steps * (1 - cfg.tail_frac)) if cfg.tail_ema else cfg.steps
    # steady-state timing: steps (warm, stop], after compile/autotune warmup and before the tail EMA starts
    stop = min(ema_start, cfg.steps) - 1
    warm = max(0, min(cfg.timing_warmup, stop - 1))
    t_first = t_warm = t_stop = None
    t0 = time.perf_counter()
    for step in range(cfg.steps):
        chunk = stream.next(cfg.batch_seqs).to(device, non_blocking=True)
        x = chunk[:-1].view(cfg.batch_seqs, cfg.seq_len)
        y = chunk[1:].view(cfg.batch_seqs, cfg.seq_len)
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=AMP):
            loss = fwd(x, y)
        loss.backward()
        mult = lr_mult(step, cfg)
        for opt in opts:
            for group in opt.param_groups:
                group["lr"] = group["initial_lr"] * mult
            opt.step()
        model.zero_grad(set_to_none=True)
        if cfg.snoo_every and (step + 1) % cfg.snoo_every == 0:
            with torch.no_grad():
                for p, s, m in zip(model.parameters(), slow, outer_m):
                    delta = p - s  # the inner steps' displacement (the negative outer gradient)
                    m.mul_(cfg.snoo_momentum).add_(delta)
                    s.add_(delta + cfg.snoo_momentum * m, alpha=cfg.snoo_lr)
                    p.copy_(s)
        if step >= ema_start:  # tail EMA of the weights (the record ends on tail averages too)
            with torch.no_grad():
                if ema is None:
                    ema = [p.detach().float().clone() for p in params]
                else:
                    torch._foreach_lerp_(ema, [p.detach().float() for p in params], 1 - cfg.tail_ema)
        if step == 0:
            sync(); t_first = time.perf_counter()
        if step == warm:
            sync(); t_warm = time.perf_counter()
        if step == stop:
            sync(); t_stop = time.perf_counter()
        if log_every and (step + 1) % log_every == 0:
            print(f"  step {step + 1}/{cfg.steps} train loss {loss.item():.4f} ({time.perf_counter() - t0:.0f}s)", flush=True)
    sync()
    t_end = time.perf_counter()
    graphs = torch._dynamo.utils.counters["stats"]["unique_graphs"] - graphs0
    val_loss = evaluate(model, val, cfg, device)
    val_loss_ema = None
    if ema is not None:
        with torch.no_grad():
            for p, e in zip(params, ema):
                p.copy_(e)
        val_loss_ema = evaluate(model, val, cfg, device)
    ms_per_step = 1000 * (t_stop - t_warm) / max(1, stop - warm)
    tokens = cfg.steps * cfg.batch_seqs * cfg.seq_len
    return dict(val_loss=val_loss, val_loss_ema=val_loss_ema,
                seconds=t_end - t0,                 # total, as before: includes compile and warmup
                first_step_s=t_first - t0,          # mostly compile
                ms_per_step=ms_per_step,            # steady state: after timing_warmup, before the tail EMA
                tokens_per_s=cfg.batch_seqs * cfg.seq_len / (ms_per_step / 1000), tokens=tokens,
                compiled_graphs=graphs, params_m=sum(p.numel() for p in model.parameters()) / 1e6)

In [ ]:
# 5. Runner and analysis. Resumable: a run is keyed by its arm, seed and scale; finished runs are skipped.
import json, math, os, statistics
import numpy as np

KEY_PREFIX = "c3"  # this notebook and code version: results of other notebooks and rounds never match


def combine(name):
    changes = {}
    for part in name.split("+"):
        changes.update(VARIANTS[part])
    return changes


def run_key(name, seed, cfg):
    return f"{KEY_PREFIX}|{name}|{seed}|{cfg.steps}|{cfg.n_layer}|{cfg.d_model}|{cfg.batch_seqs}|{cfg.seq_len}"


def load_results():
    done = {}
    if os.path.exists(RESULTS_FILE):
        for line in open(RESULTS_FILE):
            r = json.loads(line)
            done[r["key"]] = r
    return done


def run(names, seeds, base):
    """Every arm in order, all of this copy's seeds each (one compile per arm)."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    done = load_results()
    results = {}
    for name in names:
        for seed in range(FIRST_SEED, FIRST_SEED + seeds):
            cfg = base.update(**combine(name), seed=seed)
            key = run_key(name, seed, cfg)
            if key not in done:
                print(f"== {name} seed {seed}: {combine(name)}", flush=True)
                r = train(cfg, DATA_DIR, device, log_every=400)
                r.update(key=key, variant=name, seed=seed, steps=cfg.steps, torch=torch.__version__,
                         gpu=torch.cuda.get_device_name() if torch.cuda.is_available() else "cpu")
                with open(RESULTS_FILE, "a") as f:
                    f.write(json.dumps(r) + "\n")
                done[key] = r
                print(f"   val {r['val_loss']:.4f}  ema {r['val_loss_ema'] or float('nan'):.4f}  "
                      f"{r['ms_per_step']:.1f} ms/step  (first step {r['first_step_s']:.0f}s, total {r['seconds']:.0f}s)",
                      flush=True)
            if not math.isfinite(done[key]["val_loss"]):
                print(f"   {name} seed {seed} diverged: left out of the analysis", flush=True)
            results.setdefault(name, []).append(done[key])
    return results


try:
    from scipy.stats import t as _student_t
    def t_sf(x, df): return float(_student_t.sf(x, df))
except ImportError:  # normal approximation: anti-conservative at small dof
    def t_sf(x, df): return 0.5 * math.erfc(x / math.sqrt(2))


def by_seed(rs, key="val_loss"):
    return {r["seed"]: r[key] for r in rs if r.get(key) is not None and math.isfinite(r[key])}  # not diverged


def paired(results, name, ref, key="val_loss"):
    """Per-seed differences name - ref in millinats (same seed = same init of every shared parameter)."""
    a, b = by_seed(results[name], key), by_seed(results[ref], key)
    seeds = sorted(set(a) & set(b))
    d = [1000 * (a[s] - b[s]) for s in seeds]
    n = len(d)
    mean = statistics.mean(d) if d else float("nan")
    se = statistics.stdev(d) / math.sqrt(n) if n > 1 else float("nan")
    t = mean / se if n > 1 and se > 0 else float("nan")
    p = 2 * t_sf(abs(t), n - 1) if n > 1 and se > 0 else float("nan")
    return dict(n=n, mean=mean, se=se, t=t, p=p, seeds=seeds, diffs=d)


def two_way(results, names, key="val_loss"):
    """Additive model val[v, s] = mu + variant_v + seed_s + noise on the complete variant x seed grid. The seed
    effect is estimated from every variant; the noise sd is pooled over (V-1)(S-1) dof (vs n-1 for one pair).
    None with fewer than 2 seeds common to every variant."""
    seeds = sorted(set.intersection(*(set(by_seed(results[n], key)) for n in names)))
    if len(names) < 2 or len(seeds) < 2:
        return None
    Y = 1000 * np.array([[by_seed(results[n], key)[s] for s in seeds] for n in names])
    V, S = Y.shape
    resid = Y - Y.mean(1, keepdims=True) - Y.mean(0, keepdims=True) + Y.mean()
    df = (V - 1) * (S - 1)
    sigma = math.sqrt((resid ** 2).sum() / df)
    return dict(seeds=seeds, S=S, df=df, sigma=sigma, means=dict(zip(names, Y.mean(1))),
                seed_effect=dict(zip(seeds, Y.mean(0) - Y.mean())))


def contrast(tw, name, ref):
    """Two-way estimate of name - ref: (d, se, p)."""
    d, se = tw["means"][name] - tw["means"][ref], tw["sigma"] * math.sqrt(2 / tw["S"])
    return d, se, 2 * t_sf(abs(d / se), tw["df"]) if se > 0 else float("nan")


def ms_ratio(results, name, ref):
    """Steady-state ms/step of name over ref, paired by seed (each pair ran in one session, on one GPU)."""
    a, b = by_seed(results[name], "ms_per_step"), by_seed(results[ref], "ms_per_step")
    seeds = set(a) & set(b)
    return statistics.mean(a[s] / b[s] for s in seeds) if seeds else float("nan")


def paired_table(results, names, title, ref_of=lambda n: "baseline", key="val_loss"):
    rows = [n for n in names if ref_of(n) != n and ref_of(n) in results]
    if not rows:
        return
    grid = list(dict.fromkeys([ref_of(n) for n in rows] + rows))
    count = {n: len(by_seed(results[n], key)) for n in grid}  # an arm with a diverged run leaves the two-way grid
    tw = two_way(results, [n for n in grid if count[n] == max(count.values())], key) if len(grid) > 2 else None
    wn, wr = max(len(n) for n in rows), max(len(ref_of(n)) for n in rows)
    print(f"\n{title}  [{key}, millinats; d = mean per-seed difference vs ref; negative is better]")
    print(f"{'variant':>{wn}} {'ref':>{wr}} {'n':>2} {'d':>7} {'se':>5} {'p':>6} | {'2-way d':>7} {'se':>5} "
          f"{'p':>6} | {'ms/step x':>9}  per-seed d")
    for n in rows:
        ref = ref_of(n)
        s = paired(results, n, ref, key)
        in_tw = tw and {n, ref} <= set(tw["means"])
        tw_cols = "{:+7.1f} {:5.1f} {:6.3f}".format(*contrast(tw, n, ref)) if in_tw else ""
        print(f"{n:>{wn}} {ref:>{wr}} {s['n']:>2} {s['mean']:+7.1f} {s['se']:5.1f} {s['p']:6.3f} | "
              f"{tw_cols:>20} | {ms_ratio(results, n, ref):9.3f}  " + " ".join(f"{x:+.0f}" for x in s["diffs"]))
    if tw:
        print(f"2-way (variant + seed): pooled noise sd {tw['sigma']:.1f} mnat per run on {tw['df']} dof; seed effects "
              + ", ".join(f"s{s} {e:+.0f}" for s, e in tw["seed_effect"].items()))


def table(results, names, title, ref_of=lambda n: "baseline"):
    paired_table(results, names, title, ref_of)
    paired_table(results, names, title + " -- tail-EMA weights", ref_of, key="val_loss_ema")


def summary_lines(*results):
    """One compact JSON line per arm and step count, every seed listed: what to paste back (copies' lines pool)."""
    arms = {}
    for res in results:
        for name, rs in res.items():
            arms[(name, rs[0]["steps"])] = rs
    rnd = lambda x, n: None if x is None else round(x, n)
    return [json.dumps(dict(arm=name, steps=steps, first_seed=FIRST_SEED, seeds=[r["seed"] for r in rs],
                            vals=[rnd(r["val_loss"], 5) for r in rs], vals_ema=[rnd(r["val_loss_ema"], 5) for r in rs],
                            ms_per_step=[rnd(r["ms_per_step"], 2) for r in rs],
                            seconds=[rnd(r.get("seconds"), 1) for r in rs],  # total, compile included
                            first_step_s=[rnd(r.get("first_step_s"), 1) for r in rs],
                            compiled_graphs=[r.get("compiled_graphs") for r in rs],  # 1 for an arm's first seed, then 0
                            gpu=",".join(sorted({r["gpu"] for r in rs})), torch=",".join(sorted({r["torch"] for r in rs}))))
            for (name, steps), rs in arms.items()]

In [ ]:
# 6. Round 3: Canon on the record-like base REC. Arms in order of importance; each runs all of this copy's seeds
# before the next starts. For a verdict on an unfinished copy, cut ARMS to the finished arms: re-running trains nothing.
assert AMP, "needs a bf16 GPU (RTX PRO 6000, H100, A100, L4): Runtime -> Change runtime type"
BASE = Config(compile=AMP)  # torch.compile on bf16-capable GPUs
REC = "polar_express+normuon+cautious_wd+value_embeds+x0_mix+attn_gate+smear+key_offset"
VARIANTS["REC"] = combine(REC)
ARMS = ["REC", "REC+canon", "REC+canon_a", "REC+canon_c", "REC+canon_bk",  # the main questions first
        "REC+rec_rep",                                                       # REC again: the noise floor
        "REC+canon_k2", "REC+canon_k3", "REC+canon_prenorm", "REC+canon_renorm", "REC+canon_from1",
        "REC+v_shift", "REC+canon+canon_bk", "REC+trigram_hash", "REC+trigram_hash+canon"]
REF = lambda arm: "REC+trigram_hash" if arm == "REC+trigram_hash+canon" else "REC"

main = run(ARMS, SEEDS, BASE)
table(main, ARMS, "ROUND 3: every arm against REC (REC+trigram_hash+canon against REC+trigram_hash)", ref_of=REF)

In [ ]:
# 7. Long horizon: REC and REC+canon at 3x the steps (59M tokens), same seeds. Does Canon's gain shrink?
long = {}
if RUN_LONG:
    long = run(["REC", "REC+canon"], SEEDS, BASE.update(steps=3 * BASE.steps))
    table(long, ["REC", "REC+canon"], f"LONG HORIZON ({3 * BASE.steps} steps)", ref_of=lambda arm: "REC")

In [ ]:
# 8. Verdict: each arm against its reference, on the final and the tail-EMA weights; then the three questions.
def estimate(results, tw, weights, key):
    """sum_a w_a val[a] in millinats, e.g. {arm: 1, ref: -1}, over the seeds of the two-way fit tw: (d, se, p, own).
    se is the two-way one (noise pooled over every arm), or the per-seed one (S-1 dof) where larger (own=True): the
    two-way model assumes every arm reacts to a seed alike, which a structural change such as Canon need not."""
    vals = {a: by_seed(results[a], key) for a in weights}
    d_s = [1000 * sum(w * vals[a][s] for a, w in weights.items()) for s in tw["seeds"]
           if all(s in v for v in vals.values())]
    S = len(d_s)
    if S < 2:
        return float("nan"), float("nan"), float("nan"), False
    d, df, own = statistics.mean(d_s), tw["df"], False
    se = tw["sigma"] * math.sqrt(sum(w * w for w in weights.values()) / S)
    if statistics.stdev(d_s) / math.sqrt(S) > se:
        se, df, own = statistics.stdev(d_s) / math.sqrt(S), S - 1, True
    return d, se, 2 * t_sf(abs(d / se), df) if se > 0 else float("nan"), own


def call(cf, ce):
    """BETTER/WORSE: both readouts p < 0.05 in the same direction; better?/worse?: one of them, the other agreeing."""
    sig = [c for c in (cf, ce) if c[2] < 0.05]
    if not sig or cf[0] * ce[0] <= 0:
        return "mixed" if sig else "no clear effect"
    word = "BETTER" if cf[0] < 0 else "WORSE"
    return word if len(sig) == 2 else word.lower() + "?"


col = lambda c: f"{c[0]:+7.1f} {c[1]:5.1f}{'*' if c[3] else ' '} {c[2]:6.3f}"
txt = lambda c: f"{c[0]:+.1f} ± {c[1]:.1f}{'*' if c[3] else ''} (p {c[2]:.3f})"


def verdict(results, arms, ref_of, long=None):
    n = {a: len(by_seed(results[a])) for a in arms if a in results}
    arms = [a for a in n if n[a] == max(n.values())]
    if len(arms) < len(n):
        print("left out (fewer seeds than the rest: diverged or unfinished):", ", ".join(a for a in n if a not in arms))
    f, e = two_way(results, arms), two_way(results, arms, "val_loss_ema")
    if f is None or e is None:
        print("The verdict needs >= 2 seeds of every arm.")
        return
    readouts = lambda w, res=results: (estimate(res, f, w, "val_loss"), estimate(res, e, w, "val_loss_ema"))
    rows = [a for a in arms if ref_of(a) != a and ref_of(a) in arms]
    print(f"\nVERDICT on {f['S']} seeds {f['seeds']}: millinats, negative = better. se: the two-way model's (noise "
          f"pooled over all arms: {f['sigma']:.1f} final, {e['sigma']:.1f} EMA per run, {f['df']} dof), or * the "
          f"arm's own per-seed one where larger ({f['S'] - 1} dof"
          + ("; pooling the other copy's seeds, last cell, gives firmer calls" if f["S"] < 4 else "")
          + f").\n{len(rows)} comparisons: expect about {0.05 * len(rows):.1f} of them at p < 0.05 by chance alone.")
    print(f"{'arm':>24} {'vs':>16} | {'d final':>7} {'se':>6} {'p':>6} | {'d EMA':>7} {'se':>6} {'p':>6} | "
          f"{'ms/step x':>9} | verdict")
    for arm in rows:
        cf, ce = readouts({arm: 1, ref_of(arm): -1})
        note = "replicate: the noise floor" if arm.endswith("rec_rep") else call(cf, ce)
        print(f"{arm:>24} {ref_of(arm):>16} | {col(cf)} | {col(ce)} | {ms_ratio(results, arm, ref_of(arm)):9.3f} | "
              + note)
    canons = [a for a in rows if ("canon" in a or "v_shift" in a) and ref_of(a) == "REC"]
    if "REC+canon" in canons:
        print("\nWHICH FORM? Each Canon arm against REC+canon (A+C, k=4, after the norm: the paper's form)")
        for arm in canons:
            if arm != "REC+canon":
                cf, ce = readouts({arm: 1, "REC+canon": -1})
                print(f"{arm:>24} | final {txt(cf)} | EMA {txt(ce)} | {call(cf, ce)}")
        mde = 2.8 * f["sigma"] * math.sqrt(2 / f["S"])  # 80% power at p < 0.05, at the two-way se
        print(f"A difference under ~2.8 se (~{mde:.0f} mnat final here) easily goes unseen: 'no clear effect' does not "
              "make two forms equal.")
        cf, ce = readouts({"REC+canon": 1, "REC": -1})
        print(f"\n1. Does Canon survive on REC? REC+canon: {txt(cf)} final, {txt(ce)} EMA: {call(cf, ce)}")
        if "REC+trigram_hash+canon" in rows:
            cf, ce = readouts({"REC+trigram_hash+canon": 1, "REC+trigram_hash": -1})
            print(f"   ... with an n-gram table (vs REC+trigram_hash): {txt(cf)} final, {txt(ce)} EMA: {call(cf, ce)}")
            cf, ce = readouts({"REC+trigram_hash+canon": 1, "REC+trigram_hash": -1, "REC+canon": -1, "REC": 1})
            print(f"   overlap, Canon's gain with the table minus without (positive: the table takes some of it): "
                  f"{txt(cf)} final, {txt(ce)} EMA")
            if f["means"]["REC+trigram_hash"] >= f["means"]["REC"]:
                print("   but REC+trigram_hash is no better than REC: unlike the record's table (gated, at several "
                      "layers), this one does not help, so the overlap says little about the record")
        best = min(canons, key=lambda a: f["means"][a] + e["means"][a])
        print(f"2. Best form (mean of both readouts): {best}: {f['means'][best] - f['means']['REC']:+.1f} final, "
              f"{e['means'][best] - e['means']['REC']:+.1f} EMA against REC (forms within ~2 se of it are a tie)")
        drift = (f"; REC+rec_rep, REC itself later in the session, reads "
                 f"{ms_ratio(results, 'REC+rec_rep', 'REC'):.3f}: the drift" if "REC+rec_rep" in rows else "")
        print("3. Cost, steady-state ms/step against REC: "
              + ", ".join(f"{a[4:]} {ms_ratio(results, a, 'REC'):.3f}" for a in canons) + drift)
    if "REC+rec_rep" in rows:
        rep = paired(results, "REC+rec_rep", "REC")["diffs"]
        sd = math.sqrt(statistics.mean(x * x for x in rep) / 2)  # a replicate's true difference is 0
        print(f"Noise: same-seed replicate differences {' '.join(f'{x:+.1f}' for x in rep)} mnat, i.e. GPU "
              f"nondeterminism ~{sd:.1f} mnat per run; all noise sources together: {f['sigma']:.1f} per run")
    if long and "REC+canon" in long and "REC" in long and "REC+canon" in arms:
        res = {**results, "REC@3x": long["REC"], "REC+canon@3x": long["REC+canon"]}
        far = readouts({"REC+canon@3x": 1, "REC@3x": -1}, res)
        change = readouts({"REC+canon@3x": 1, "REC@3x": -1, "REC+canon": -1, "REC": 1}, res)
        print(f"Long horizon: REC+canon - REC at 3x the steps {txt(far[0])} final, {txt(far[1])} EMA; the change from "
              f"the base steps {txt(change[0])} final, {txt(change[1])} EMA (positive: the gain shrinks with tokens)")


verdict(main, ARMS, REF, long)

In [ ]:
# 9. Summary: copy this whole output back to Claude (from both copies if you ran two).
SUMMARY = "\n".join(summary_lines(main, long))
print(SUMMARY)

In [ ]:
# 10. Optional: pool both copies. Paste the other copy's Summary output between the quotes and run this cell to get
# the verdict on every seed. (A copy's session and GPU are part of its seeds' effects, which the two-way model removes.)
OTHER_COPY = """
"""


def from_summary(text, steps):
    """{arm: [run, ...]} with the fields the analysis reads, from summary lines at this step count."""
    out = {}
    for line in text.splitlines():
        if line.strip().startswith('{"arm"'):
            s = json.loads(line)
            if s["steps"] == steps:
                out.setdefault(s["arm"], []).extend(
                    dict(seed=seed, val_loss=v, val_loss_ema=ema, ms_per_step=ms)
                    for seed, v, ema, ms in zip(s["seeds"], s["vals"], s["vals_ema"], s["ms_per_step"]))
    return out


seeds_in = lambda text: {seed for line in text.splitlines() if line.strip().startswith('{"arm"')
                         for seed in json.loads(line)["seeds"]}
common = seeds_in(SUMMARY) & seeds_in(OTHER_COPY)
if common:  # the same (arm, seed) twice: one copy's runs would silently replace the other's
    print(f"Both summaries have seeds {sorted(common)}: not pooled. Set FIRST_SEED = 2 in one of the two copies.")
elif OTHER_COPY.strip():
    both = SUMMARY + "\n" + OTHER_COPY
    verdict(from_summary(both, BASE.steps), ARMS, REF, from_summary(both, 3 * BASE.steps))
else:
    print("To pool: paste the other copy's summary into OTHER_COPY and run this cell again.")